In [ ]:
# Last Update: 6/30/2026 ; 8:27 am

In [1]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs = 100

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, _, _, _ = model(eeg_b, nirs_b)
            
            loss = criterion(pred, lbl_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    all_preds, all_true = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, _, _, _ = model(eeg_b, nirs_b)
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")


Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
# Last Update: 7/3/2026 ; 11 : 24 pm

In [1]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, _, _, _ = model(eeg_b, nirs_b)
            
            loss = criterion(pred, lbl_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    all_preds, all_true = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, _, _, _ = model(eeg_b, nirs_b)
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
# Last Update: 7/3/2026 ; 12 : 00 pm
#TSFNet_model_FIXED_v3_confmatrix_ablation

In [2]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, _, _, _ = model(eeg_b, nirs_b)
            
            loss = criterion(pred, lbl_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)  # CHANGED: no longer discarding eeg_out/fnirs_out/fusion_out
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
# Last Update: 7/3/2026 ; 12 : 12 pm
#Tsfnet model fixed v4 weights fusiononly

In [3]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, _, _, _ = model(eeg_b, nirs_b)
            
            loss = criterion(pred, lbl_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)  # CHANGED: no longer discarding eeg_out/fnirs_out/fusion_out
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
# Last Update: 7/3/2026 ; 12 :    pm
#Tsfnet model fixed v5 ordinal

In [4]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# ORDINAL CLASSIFICATION UTILITIES (NEW)
# NEW (ordinal classification): implements the "extended binary classification" decomposition for
# ordinal targets (Frank, E. & Hall, M., 2001, "A simple approach to ordinal classification", ECML;
# see also da Costa & Cardoso, 2005, ECML, and Cheng et al., IEEE IJCNN, both already referenced
# earlier in this thesis's literature review as standard ordinal-classification baselines). The
# 3 n-back classes (0-back=0, 2-back=1, 3-back=2) have a natural order, so instead of standard
# K-way softmax classification (which treats a 3-back-predicted-as-0-back error identically to a
# 3-back-predicted-as-2-back error), this decomposes the problem into K-1=2 binary "threshold"
# questions:
#   threshold 0: "is the true class > 0-back?"  (i.e. is it 2-back OR 3-back?)
#   threshold 1: "is the true class > 2-back?"  (i.e. is it 3-back?)
# Each classifier head (EEG-only, fNIRS-only, fusion) now outputs 2 raw logits (num_classes-1)
# instead of 3, trained with independent binary cross-entropy per threshold. At inference, the
# predicted rank is the number of thresholds whose sigmoid probability exceeds 0.5 -- e.g. if both
# thresholds fire, predicted rank=2 (3-back); if only the first fires, rank=1 (2-back); if neither
# fires, rank=0 (0-back). This makes errors ordinally "close" (e.g. 3-back predicted as 2-back)
# cheaper than errors that violate the order, and was motivated directly by the confusion-matrix
# evidence showing a systematic bias toward the middle class (2-back) rather than a symmetric,
# order-agnostic error pattern.
def ordinal_targets(y, num_classes):  # NEW
    """Builds (batch, num_classes-1) binary threshold targets from integer class labels y."""
    batch_size = y.size(0)  # NEW
    n_thresholds = num_classes - 1  # NEW
    targets = torch.zeros(batch_size, n_thresholds, device=y.device, dtype=torch.float32)  # NEW
    for k in range(n_thresholds):  # NEW
        targets[:, k] = (y > k).float()  # NEW: threshold k asks "is true rank > k?"
    return targets  # NEW


def ordinal_loss(logits, y, num_classes):  # NEW
    """Binary cross-entropy summed over all K-1 threshold tasks (Frank & Hall, 2001)."""
    targets = ordinal_targets(y, num_classes)  # NEW
    return F.binary_cross_entropy_with_logits(logits, targets)  # NEW


def ordinal_predict(logits):  # NEW
    """Decodes (batch, num_classes-1) threshold logits into integer class predictions by counting
    how many thresholds are exceeded (sigmoid probability > 0.5)."""
    probs = torch.sigmoid(logits)  # NEW
    preds = (probs > 0.5).sum(dim=1)  # NEW: number of thresholds exceeded = predicted rank
    return preds  # NEW



class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        # self.fc_eeg = nn.Linear(64, num_classes)
        # self.fc_fnirs = nn.Linear(64, num_classes)
        # self.fc_fusion = nn.Linear(64, num_classes)
        # FIX (ordinal classification): output dimension changed from num_classes (3, standard
        # softmax classification) to num_classes-1 (2, ordinal "threshold" logits -- see
        # ordinal_targets/ordinal_loss/ordinal_predict defined above this class for the full
        # rationale). This is the ONLY architectural change required for ordinal classification;
        # all conv/EFGF/CAFE layers upstream are completely unchanged, as planned.
        self.num_classes = num_classes  # NEW: stored for use in forward() combination and elsewhere
        self.fc_eeg = nn.Linear(64, num_classes - 1)      # NEW: ordinal output (K-1 threshold logits)
        self.fc_fnirs = nn.Linear(64, num_classes - 1)    # NEW
        self.fc_fusion = nn.Linear(64, num_classes - 1)   # NEW
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # FIX (ordinal classification): standard K-way CrossEntropyLoss treats all misclassifications
    # equally (0-back predicted as 3-back is penalized the same as 3-back predicted as 2-back),
    # which does not reflect the ordinal structure of cognitive load levels (0 < 2 < 3-back).
    # Replaced with the ordinal_loss() function defined above the model class, which trains the
    # num_classes-1=2 threshold logits from each classifier head via independent binary
    # cross-entropy (Frank & Hall, 2001). NUM_CLASSES is stored once here for reuse throughout
    # the training/evaluation loops below.
    NUM_CLASSES = 3  # NEW
    criterion = lambda logits, y: ordinal_loss(logits, y, NUM_CLASSES)  # NEW: ordinal (threshold-based) loss replaces CrossEntropyLoss

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, _, _, _ = model(eeg_b, nirs_b)
            
            loss = criterion(pred, lbl_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            # train_correct += (pred.argmax(1) == lbl_b).sum().item()
            # FIX (ordinal classification): pred now holds num_classes-1=2 raw threshold logits per
            # sample, not num_classes=3 class-score logits, so argmax(1) no longer produces a valid
            # class index. ordinal_predict() decodes the correct class by counting how many
            # thresholds are exceeded (see function definition above the model class).
            train_correct += (ordinal_predict(pred) == lbl_b).sum().item()  # NEW: ordinal decoding replaces argmax
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                # val_correct += (pred.argmax(1) == lbl_b).sum().item()
                # FIX (ordinal classification): same reasoning as the train-loop fix above.
                val_correct += (ordinal_predict(pred) == lbl_b).sum().item()  # NEW: ordinal decoding replaces argmax
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)  # CHANGED: no longer discarding eeg_out/fnirs_out/fusion_out
            # all_preds_final.extend(pred.argmax(1).cpu().numpy())
            # all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())
            # all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())
            # all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())
            # FIX (ordinal classification): all four outputs (pred/eeg_out/fnirs_out/fusion_out) now
            # hold num_classes-1=2 raw threshold logits per sample, not 3-class scores, so argmax(1)
            # no longer decodes a valid class. ordinal_predict() applies the same threshold-counting
            # decoding used during training/validation.
            all_preds_final.extend(ordinal_predict(pred).cpu().numpy())         # NEW: ordinal decoding replaces argmax
            all_preds_eeg.extend(ordinal_predict(eeg_out).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(ordinal_predict(fnirs_out).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(ordinal_predict(fusion_out).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set)...
--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]

Sc

In [ ]:
# Last Update: 7/3/2026 ; 5 : 08   am
#Tsfnet model fixed v6 paper loss

In [1]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced  # NEW: added eeg_feat, hybrid, nirs_feat, enhanced

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    # optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    # FIX (overfitting -- items 2 & 3 of the plan): the previous training run showed severe
    # overfitting (train acc 96% vs val acc stuck at ~34-41% from epoch ~5 onward), consistent
    # with regularization being too weak for the amount of inter-subject variability in this
    # subject-independent 3-class n-back setup. Two changes address this directly:
    # (2) weight_decay raised from 5e-4 to 2e-2 (a 40x increase) to apply much stronger L2
    #     regularization on the ~80k model parameters, discouraging the model from fitting
    #     subject-specific noise patterns in the training set.
    # (3) learning rate lowered from 5e-4 to 2e-4, and the ReduceLROnPlateau patience shortened
    #     from 10 to 5 epochs, so the scheduler reacts faster once validation accuracy stops
    #     improving (which, per the previous run, happened almost immediately -- best val was
    #     reached at epoch 5).
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: lr 5e-4->2e-4, weight_decay 5e-4->2e-2
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW: patience 10->5
    # criterion = nn.CrossEntropyLoss()
    # NEW (TSFNet paper Eq. 5-13 -- multi-branch + regularization loss): replacing the single
    # criterion on the final prediction alone with the full loss design from the paper. See the
    # pearson_correlation_loss() docstring and the forward()-return-value comment above for the
    # full rationale. LAMBDA_EEG and LAMBDA_FNIRS follow the paper's Eq. 6 notation (lambda_eeg,
    # lambda_fnirs); the paper states these were tuned via a parameter search and does not report
    # their exact values, so 0.3 is used here as a reasonable default that gives each auxiliary
    # branch meaningful but subordinate weight relative to the main prediction loss L_pred
    # (coefficient 1) -- this value is a thesis-specific choice and should ideally be tuned via the
    # validation set if time permits. L_fusion, L_efgf, and L_cafe use coefficient 1, exactly as in
    # the paper's Eq. 6 and Eq. 5 (no separate lambda given for these terms).
    criterion = nn.CrossEntropyLoss()  # NEW: still CrossEntropyLoss per-branch, just applied to 4 branches instead of 1 (see loss computation in the training loop below)
    LAMBDA_EEG = 0.3     # NEW: Eq. 6 lambda_eeg -- thesis-specific default, paper does not report the tuned value
    LAMBDA_FNIRS = 0.3   # NEW: Eq. 6 lambda_fnirs -- thesis-specific default, paper does not report the tuned value
    LAMBDA_EFGF_REG = 1.0   # NEW: Eq. 5 weight for L_efgf (paper uses implicit coefficient 1)
    LAMBDA_CAFE_REG = 1.0   # NEW: Eq. 5 weight for L_cafe (paper uses implicit coefficient 1)

    best_val_acc = 0.0
    epochs = 100

    # NEW (overfitting fix, item 1 -- real early stopping): the previous run kept training for
    # the full 100 epochs even though best_val_acc was reached at epoch 5 and never improved
    # again, wasting ~95 epochs of compute and letting the model overfit further (train acc kept
    # climbing to 96% with no benefit to val/test). EARLY_STOP_PATIENCE tracks how many epochs
    # have passed since the last val-accuracy improvement; once that count reaches the patience
    # threshold, the training loop is broken early (not just "best weights saved", but training
    # itself stops), matching the user's explicit request.
    EARLY_STOP_PATIENCE = 15  # NEW: stop if val acc does not improve for 15 consecutive epochs
    epochs_no_improve = 0  # NEW: counter of consecutive epochs without val-acc improvement

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            # pred, _, _, _ = model(eeg_b, nirs_b)
            # 
            # loss = criterion(pred, lbl_b)
            # NEW (TSFNet paper Eq. 5-13): full multi-branch classification loss (Eq. 6) plus EFGF
            # (Eq. 12) and CAFE (Eq. 13) regularization losses, replacing the single-term loss above
            # that only supervised the final fused prediction. eeg_feat_map/efgf_attn_map are the
            # analogues used for L_efgf; nirs_feat_map/cafe_enhanced_map are the analogues used for
            # L_cafe (see forward()'s return-value comment for the exact correspondence).
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs

            L_pred = criterion(pred, lbl_b)          # NEW: Eq. 10 (generalized to 3-class CrossEntropyLoss)
            L_eeg = criterion(eeg_out, lbl_b)        # NEW: Eq. 7 (generalized to 3-class CrossEntropyLoss)
            L_fnirs = criterion(fnirs_out, lbl_b)    # NEW: Eq. 8 (generalized to 3-class CrossEntropyLoss)
            L_fusion = criterion(fusion_out, lbl_b)  # NEW: Eq. 9 (generalized to 3-class CrossEntropyLoss)
            L_class = L_pred + LAMBDA_EEG * L_eeg + LAMBDA_FNIRS * L_fnirs + L_fusion  # NEW: Eq. 6

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)      # NEW: Eq. 12
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW: Eq. 13

            loss = L_class + LAMBDA_EFGF_REG * L_efgf + LAMBDA_CAFE_REG * L_cafe  # NEW: Eq. 5 (total loss)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                # pred, _, _, _ = model(eeg_b, nirs_b)
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW: unpack all 8 outputs, only final pred needed for val accuracy
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        # if val_acc > best_val_acc:
        #     best_val_acc = val_acc
        #     torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
        # NEW (overfitting fix, item 1, continued): same "save best weights" logic as before, plus
        # tracking epochs_no_improve to drive the early-stop break condition below.
        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))  # NEW
            epochs_no_improve = 0  # NEW: reset counter on improvement
        else:  # NEW
            epochs_no_improve += 1  # NEW: no improvement this epoch
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

        # NEW (overfitting fix, item 1, continued): actually stop training -- not just keep the
        # best checkpoint -- once val accuracy has not improved for EARLY_STOP_PATIENCE epochs.
        if epochs_no_improve >= EARLY_STOP_PATIENCE:  # NEW
            print(f"\nEarly stopping triggered at epoch {epoch} "  # NEW
                  f"(no val-acc improvement for {EARLY_STOP_PATIENCE} consecutive epochs). "  # NEW
                  f"Best Val Acc: {best_val_acc*100:.2f}% (epoch {epoch - epochs_no_improve}).")  # NEW
            break  # NEW: exit the training loop early

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    # all_preds, all_true = [], []
    # with torch.no_grad():
    #     for eeg_b, nirs_b, lbl_b in test_loader:
    #         eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
    #         pred, _, _, _ = model(eeg_b, nirs_b)
    #         all_preds.extend(pred.argmax(1).cpu().numpy())
    #         all_true.extend(lbl_b.cpu().numpy())
    #
    # print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    # print("\nClassification Report:")
    # print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    # NEW (user request -- confusion matrix + zero-retraining ablation): TSFNet_With_fNIRS_2D.forward()
    # already returns FOUR outputs every call: final_out (the fused prediction, i.e. what "pred" was
    # before), eeg_out (EEG-branch-only classifier head), fnirs_out (fNIRS-branch-only classifier
    # head), and fusion_out (the EFGF+CAFE fused-feature classifier head, before the final weighted
    # combination w1*eeg_out + w2*fnirs_out + w3*fusion_out). This means an EEG-only vs fNIRS-only vs
    # fusion-only ablation can be computed for FREE from the already-trained best model, with zero
    # additional training, simply by collecting and evaluating each of these four outputs separately
    # on the test set instead of discarding eeg_out/fnirs_out/fusion_out (as the original code did
    # with the "_, _, _" placeholders).
    all_preds_final, all_preds_eeg, all_preds_fnirs, all_preds_fusion, all_true = [], [], [], [], []  # NEW
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            # pred, eeg_out, fnirs_out, fusion_out = model(eeg_b, nirs_b)
            pred, eeg_out, fnirs_out, fusion_out, _, _, _, _ = model(eeg_b, nirs_b)  # CHANGED: forward() now returns 8 values; the 4 extra feature maps (used only for EFGF/CAFE regularization loss during training) are ignored here
            all_preds_final.extend(pred.argmax(1).cpu().numpy())         # NEW
            all_preds_eeg.extend(eeg_out.argmax(1).cpu().numpy())        # NEW: EEG-branch-only prediction
            all_preds_fnirs.extend(fnirs_out.argmax(1).cpu().numpy())    # NEW: fNIRS-branch-only prediction
            all_preds_fusion.extend(fusion_out.argmax(1).cpu().numpy())  # NEW: EFGF+CAFE fused-feature-only prediction
            all_true.extend(lbl_b.cpu().numpy())

    class_names = ['0-back', '2-back', '3-back']  # NEW

    print(f"\nOverall Model Test Accuracy (final fused prediction): {accuracy_score(all_true, all_preds_final)*100:.2f}%")  # CHANGED: clarified label
    print("\nClassification Report (final fused prediction):")  # CHANGED: clarified label
    print(classification_report(all_true, all_preds_final, target_names=class_names))  # CHANGED: uses all_preds_final

    # NEW: full confusion matrix for the final fused prediction, to see exactly where errors
    # concentrate (e.g. whether 2-back vs 3-back confusion dominates, as predicted by the literature
    # reviewed for this thesis -- Cao et al. 2022, Ghasimi & Shamekhi 2024, Aghajani et al. 2017, etc.)
    cm = confusion_matrix(all_true, all_preds_final)  # NEW
    print("\nConfusion Matrix (rows=true, cols=predicted):")  # NEW
    header = "        " + "".join([f"{c:>10}" for c in class_names])  # NEW
    print(header)  # NEW
    for i, row in enumerate(cm):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    # NEW: zero-retraining ablation -- accuracy of each individual branch on the SAME trained model,
    # to see whether the EFGF/CAFE fusion mechanism is actually adding value over a single modality
    # alone, or whether one branch is doing most/all of the work.
    print("\nZero-Retraining Ablation (individual classifier heads of the SAME trained model):")  # NEW
    print(f"  EEG-branch-only accuracy:          {accuracy_score(all_true, all_preds_eeg)*100:.2f}%")  # NEW
    print(f"  fNIRS-branch-only accuracy:         {accuracy_score(all_true, all_preds_fnirs)*100:.2f}%")  # NEW
    print(f"  EFGF+CAFE fused-feature accuracy:   {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print(f"  Final weighted-combination accuracy: {accuracy_score(all_true, all_preds_final)*100:.2f}%  (w1*eeg + w2*fnirs + w3*fusion)")  # NEW

    # NEW: print the model's LEARNED final-combination weights (w1, w2, w3 in the model definition
    # are raw nn.Parameter scalars; the model applies them via softmax before combining, exactly as
    # in the forward() method: w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)).
    # This tells us how much weight the model itself assigned to each branch after training, which
    # helps explain why fusion_out alone (41.35% in the previous run) can outperform the weighted
    # final_out (40.75%): if w1/w2 (eeg/fnirs) are not small enough, they dilute the stronger
    # fusion_out signal with weaker, near-chance-level eeg_out/fnirs_out predictions.
    with torch.no_grad():  # NEW
        learned_weights = F.softmax(torch.stack([model.w1, model.w2, model.w3]), dim=0).cpu().numpy()  # NEW
    print(f"\nLearned final-combination weights (after softmax): "  # NEW
          f"w_eeg={learned_weights[0]:.3f}, w_fnirs={learned_weights[1]:.3f}, w_fusion={learned_weights[2]:.3f}")  # NEW

    # NEW (user request): direct experiment -- evaluate using fusion_out ALONE as the model's final
    # prediction (bypassing the learned weighted combination entirely), using predictions already
    # collected above (all_preds_fusion), so this requires NO retraining. This directly tests whether
    # dropping the eeg_out/fnirs_out contribution from the final decision improves test performance,
    # motivated by the previous run showing fusion_out (41.35%) outperforming the weighted
    # final_out (40.75%).
    print("\n" + "=" * 70)  # NEW
    print("DIRECT EXPERIMENT: Using fusion_out (EFGF+CAFE) ALONE as final prediction")  # NEW
    print("(bypassing the learned w1*eeg + w2*fnirs + w3*fusion combination)")  # NEW
    print("=" * 70)  # NEW
    print(f"\nfusion_out-only Test Accuracy: {accuracy_score(all_true, all_preds_fusion)*100:.2f}%")  # NEW
    print("\nClassification Report (fusion_out only):")  # NEW
    print(classification_report(all_true, all_preds_fusion, target_names=class_names))  # NEW

    cm_fusion = confusion_matrix(all_true, all_preds_fusion)  # NEW
    print("Confusion Matrix -- fusion_out only (rows=true, cols=predicted):")  # NEW
    print(header)  # NEW: reuse the same column header printed for the final-prediction confusion matrix above
    for i, row in enumerate(cm_fusion):  # NEW
        row_str = "".join([f"{v:>10}" for v in row])  # NEW
        print(f"{class_names[i]:>8}{row_str}")  # NEW

    improvement = (accuracy_score(all_true, all_preds_fusion) - accuracy_score(all_true, all_preds_final)) * 100  # NEW
    print(f"\nfusion_out-only vs weighted-final accuracy difference: {improvement:+.2f} percentage points")  # NEW
    if improvement > 0:  # NEW
        print("--> fusion_out ALONE outperforms the learned weighted combination on this test run.")  # NEW
        print("    This suggests architecturally simplifying the final decision to rely on fusion_out")  # NEW
        print("    alone (or re-weighting w1/w2/w3 more aggressively toward fusion) may be worthwhile.")  # NEW
    else:  # NEW
        print("--> The learned weighted combination performs at least as well as fusion_out alone")  # NEW
        print("    on this test run; no architectural change indicated from this experiment.")  # NEW

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...


ValueError: could not broadcast input array from shape (1026,16,16,600) into shape (1026,28,600)

In [ ]:
# Last Update: 7/3/2026 ; 5 : 25   pm
# Tsfnet optuna search

In [2]:
!pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------  2.1/2.1 MB 10.7 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 11.0 MB/s  0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------------- -------------------------- 2/6 [colorlog]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import optuna
print("optuna version:", optuna.__version__)

optuna version: 4.9.0


In [4]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import gc
import math
# import numpy as np
# FIX (Problem 3, part 1): 'random' module was used below (random.shuffle) but never imported,
# which would raise NameError: name 'random' is not defined the moment the script runs.
import random  # NEW: was missing, required for random.shuffle below
import numpy as np  # NEW: moved down here only for readability, same import as before
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
# NEW (user request): confusion_matrix needed for the detailed error-pattern analysis, and
# accuracy_score is reused per-branch for the EEG-only / fNIRS-only ablation below.
from sklearn.metrics import confusion_matrix  # NEW
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# FIX (Problem 3, part 2): no fixed seed was set before shuffling the subject list. Without a
# fixed seed, every run of this script produces a DIFFERENT train/val/test subject split, which
# means reported accuracy numbers cannot be reproduced later (by the author, an advisor, or a
# reviewer) -- a methodological requirement for thesis-level reporting. Fixing random.seed (for
# the subject shuffle), torch.manual_seed (for model weight initialization and dropout), and
# np.random.seed (for any numpy-side randomness) makes the entire train/val/test split AND the
# model's initial state reproducible across runs.
RANDOM_SEED = 42  # NEW: fixed seed for reproducible subject-level train/val/test split and model init
random.seed(RANDOM_SEED)  # NEW
torch.manual_seed(RANDOM_SEED)  # NEW: also fixes PyTorch's own random state (weight init, dropout)
np.random.seed(RANDOM_SEED)  # NEW

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    # def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
    # FIX (Problem 1): min_windows_threshold=2000 is higher than every subject's actual window
    # count after preprocessing (observed range across all 26 subjects: 752-1026, per the
    # quality-check summary table). With threshold=2000, EVERY subject gets skipped and
    # self.total_windows stays 0, raising "No valid preprocessed data found" for all three splits.
    # Lowering the threshold to 500 still filters out any genuinely broken/near-empty subject
    # folder while keeping all 26 currently valid subjects (minimum observed: 752, VP004).
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=500):  # FIX: was 2000, no subject reaches that (observed range is 752-1026)
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)

# NEW (TSFNet paper Eq. 6-13 -- multi-branch classification loss + EFGF/CAFE regularization loss):
# The original TSFNet paper (Zhang, Yin, Yuan, Sensors 2025, 25, 6111) does NOT train the model
# using cross-entropy on the final fused prediction alone. Its total loss (Eq. 5) is
# L = L_class + L_efgf + L_cafe, where L_class (Eq. 6) itself sums FOUR terms:
#   L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
# i.e. the EEG branch, fNIRS branch, and fusion branch each receive their OWN direct classification
# supervision (not just indirect gradient through the final weighted combination, as the previous
# version of this code did with `loss = criterion(pred, lbl_b)` alone). The paper's Eq. 7-10 use
# binary cross-entropy because their MI/MA/WG tasks are binary; here they are generalized to
# standard multi-class CrossEntropyLoss (equivalent formula, just softmax over C=3 classes instead
# of 2) since this thesis's n-back task has 3 classes (0-/2-/3-back).
#
# The EFGF regularization loss (Eq. 12) and CAFE regularization loss (Eq. 13) use the Pearson
# Correlation Coefficient (PCC, Eq. 11) between intermediate feature maps to force the EFGF/CAFE
# layers to genuinely learn cross-modal correlation, rather than an arbitrary combination. This
# architecture (Conv1D/Conv3D + pooled 64-d feature vectors) does not produce the same-shaped
# spatial H x W x T feature maps as the paper's full 3D-CNN branches, so the PCC is computed
# between the closest available intermediate representations for each layer (see forward(), where
# eeg_feat / efgf_attention_map and nirs_feat / cafe_enhanced_feat are returned specifically for
# this purpose): this is a faithful generalization of the paper's regularization intent to this
# thesis's simplified branch architecture, not a verbatim reproduction of its exact tensor shapes.
def pearson_correlation_loss(a, b, eps=1e-8):  # NEW
    """
    Computes 1 - PCC(a, b) as a regularization loss, following TSFNet Eq. 11-13.
    a, b: tensors of shape (batch, features). PCC is computed treating the whole batch of
    feature vectors as the (N, M) double sum in Eq. 11 (N=batch, M=features), matching the
    paper's formulation where PCC is computed over all samples and all feature-map elements
    jointly, not per-sample.
    """
    a_flat = a.reshape(-1)  # NEW: flatten (batch, features) -> (batch*features,), matching the paper's double sum over N and M
    b_flat = b.reshape(-1)  # NEW
    a_centered = a_flat - a_flat.mean()  # NEW
    b_centered = b_flat - b_flat.mean()  # NEW
    numerator = (a_centered * b_centered).sum()  # NEW
    denominator = torch.sqrt((a_centered ** 2).sum() * (b_centered ** 2).sum()) + eps  # NEW
    pcc = numerator / denominator  # NEW: ranges [-1, 1]
    return 1.0 - pcc  # NEW: Eq. 12/13 form -- loss is minimized as PCC -> 1 (perfect positive correlation)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        # FIX (Problem 2): nirs_seq_proj used to be created INSIDE forward() as
        # nn.Linear(32, 64).to(nirs.device)(nirs_seq), which instantiates a brand-new Linear layer
        # with RANDOM, UNTRAINED weights on every single forward pass. Consequences: (1) this layer
        # is never registered in self.parameters(), so the optimizer NEVER updates it -- its weights
        # stay random forever; (2) the same input produces a DIFFERENT output every forward call,
        # injecting pure noise into the CAFE block, which is the paper's core fusion-refinement
        # mechanism. Moving it here to __init__ makes it a persistent, trainable layer like every
        # other layer in the model.
        self.nirs_seq_proj = nn.Linear(32, 64)  # NEW: moved from forward() -- must be a persistent, trainable layer
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        # nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq)
        # BUG: was re-created every forward call with random, untrained weights (see __init__ comment)
        nirs_seq_proj = self.nirs_seq_proj(nirs_seq)  # FIX: uses the persistent, trainable layer defined in __init__
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        # return final_out, eeg_out, fnirs_out, fusion_out
        # NEW (TSFNet paper Eq. 12-13 support): also return the intermediate feature maps needed to
        # compute the EFGF and CAFE regularization losses outside this function (in the training
        # loop), without changing the internal computation above.
        #   - EFGF regularization (Eq. 12) needs the "average-pooled EEG feature map" (eeg_feat,
        #     already pooled to a 64-d vector by self.eeg_pool/self.eeg_fc above) and the
        #     "EEG-fNIRS-guided attention map" produced by the EFGF layer -- the closest analogue
        #     in this simplified architecture is `gate` (the sigmoid attention gate computed from
        #     the EEG+fNIRS hybrid feature just above), which plays the same functional role as
        #     Phi in the paper (an attention map derived jointly from EEG and fNIRS features).
        #   - CAFE regularization (Eq. 13) needs the "effective fNIRS features" (nirs_feat) and the
        #     "enhanced fusion features" -- the closest analogue is `enhanced` (the output of the
        #     CAFE cross-attention block, before the final gate2-weighted combination with fused_feat).
        # return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, gate, nirs_feat, enhanced
        # FIX (shape mismatch): `gate` has shape (batch, 1) since self.efgf_gate = nn.Linear(64, 1),
        # while `eeg_feat` has shape (batch, 64). Element-wise PCC (Eq. 12) requires two tensors of
        # matching shape. `hybrid` (shape (batch, 64), the EEG-fNIRS-guided combined representation
        # computed just above as alpha*eeg_p + (1-alpha)*nirs_p) is used instead: it is the direct
        # joint EEG+fNIRS representation from which `gate` itself is derived, making it a shape-valid
        # and conceptually sound analogue of the paper's attention map Phi (also derived jointly from
        # both modalities) for this simplified architecture.
        return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, hybrid, nirs_feat, enhanced  # NEW: added eeg_feat, hybrid, nirs_feat, enhanced


# NEW (Optuna-based hyperparameter search for LAMBDA_EEG / LAMBDA_FNIRS):
# Rationale: the previous training run (with LAMBDA_EEG=LAMBDA_FNIRS=0.3, a manual guess) showed
# an unbalanced effect -- the fNIRS branch improved (37.38% -> 39.22%) but the EEG branch got
# WORSE (36.91% -> 33.91%, essentially chance level). This suggests the two lambdas should NOT
# necessarily be equal, and a proper search (rather than another manual guess) is needed to find
# values that genuinely improve validation accuracy for the FINAL fused prediction (the quantity
# that actually matters for the thesis's reported result), not just one branch in isolation.
#
# Method: Optuna's default sampler (TPE -- Tree-structured Parzen Estimator; Bergstra, Bardenet,
# Bengio & Kegl, 2011, "Algorithms for Hyper-Parameter Optimization", NeurIPS) is used instead of
# grid or random search, since it adaptively concentrates sampling in promising regions of the
# search space based on the results of previous trials, giving a more accurate optimum for a given
# computational budget (Bergstra & Bengio, 2012, "Random Search for Hyper-Parameter Optimization",
# JMLR, for the random-vs-grid comparison that motivates using random/Bayesian methods over grid
# search in continuous hyperparameter spaces).
#
# IMPORTANT: the objective is evaluated on VALIDATION accuracy only -- the test set is NEVER
# touched during this search, to avoid any data leakage into the final reported test accuracy.
# The test set is only used once, at the very end, to evaluate the single best configuration found.
import time  # NEW
import optuna  # NEW: pip install optuna if not already available

N_TRIALS = 20  # NEW: number of Optuna trials to run; increase if more time is available (see time estimate printed per trial)
SEARCH_EARLY_STOP_PATIENCE = 10  # NEW: shorter patience than the final single-run training (15) to keep each trial faster during the search itself
SEARCH_MAX_EPOCHS = 60  # NEW: hard cap per trial so a single unlucky trial cannot run for the full 100 epochs


def build_dataloaders():  # NEW: load train/val ONCE, reused across all Optuna trials (avoids re-scanning/re-streaming subjects to disk on every trial, which is the expensive part)
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    return train_dataset, val_dataset, train_loader, val_loader


def train_one_config(lambda_eeg, lambda_fnirs, train_loader, val_loader, device,
                      lambda_efgf_reg=1.0, lambda_cafe_reg=1.0,
                      max_epochs=SEARCH_MAX_EPOCHS, patience=SEARCH_EARLY_STOP_PATIENCE,
                      verbose=False):  # NEW: reusable training routine, called once per Optuna trial and once more for the final chosen configuration
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)  # NEW: fresh model per trial (no weight reuse across trials)
    optimizer = optim.AdamW(model.parameters(), lr=0.0002, weight_decay=2e-2)  # NEW: same optimizer settings as the main training script
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)  # NEW
    criterion = nn.CrossEntropyLoss()  # NEW

    best_val_acc = 0.0  # NEW
    epochs_no_improve = 0  # NEW

    for epoch in range(1, max_epochs + 1):  # NEW
        model.train()  # NEW
        for eeg_b, nirs_b, lbl_b in train_loader:  # NEW
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)  # NEW

            optimizer.zero_grad()  # NEW
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat_map, efgf_attn_map, nirs_feat_map, cafe_enhanced_map = model(eeg_b, nirs_b)  # NEW

            L_pred = criterion(pred, lbl_b)  # NEW
            L_eeg = criterion(eeg_out, lbl_b)  # NEW
            L_fnirs = criterion(fnirs_out, lbl_b)  # NEW
            L_fusion = criterion(fusion_out, lbl_b)  # NEW
            L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion  # NEW

            L_efgf = pearson_correlation_loss(eeg_feat_map, efgf_attn_map)  # NEW
            L_cafe = pearson_correlation_loss(nirs_feat_map, cafe_enhanced_map)  # NEW

            loss = L_class + lambda_efgf_reg * L_efgf + lambda_cafe_reg * L_cafe  # NEW
            loss.backward()  # NEW
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # NEW
            optimizer.step()  # NEW

        # validation
        model.eval()  # NEW
        val_correct = 0  # NEW
        with torch.no_grad():  # NEW
            for eeg_b, nirs_b, lbl_b in val_loader:  # NEW
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)  # NEW
                pred, _, _, _, _, _, _, _ = model(eeg_b, nirs_b)  # NEW
                val_correct += (pred.argmax(1) == lbl_b).sum().item()  # NEW

        val_acc = val_correct / len(val_loader.dataset)  # NEW
        scheduler.step(val_acc)  # NEW

        if val_acc > best_val_acc:  # NEW
            best_val_acc = val_acc  # NEW
            epochs_no_improve = 0  # NEW
        else:  # NEW
            epochs_no_improve += 1  # NEW

        if verbose and (epoch % 5 == 0 or epoch == 1):  # NEW
            print(f"    epoch {epoch:3d} | val_acc={val_acc*100:.2f}% | best={best_val_acc*100:.2f}%")  # NEW

        if epochs_no_improve >= patience:  # NEW
            break  # NEW

    return best_val_acc, model  # NEW: return the trained model too, so the final best trial's weights can optionally be reused/saved


def objective(trial):  # NEW: Optuna objective function -- returns the metric to MAXIMIZE (validation accuracy)
    lambda_eeg = trial.suggest_float('lambda_eeg', 0.05, 0.8)  # NEW: continuous search range
    lambda_fnirs = trial.suggest_float('lambda_fnirs', 0.05, 0.8)  # NEW

    t0 = time.time()  # NEW
    best_val_acc, _ = train_one_config(  # NEW
        lambda_eeg, lambda_fnirs,
        GLOBAL_TRAIN_LOADER, GLOBAL_VAL_LOADER, GLOBAL_DEVICE,
        verbose=False
    )
    elapsed_min = (time.time() - t0) / 60  # NEW

    print(f"Trial {trial.number:3d} | lambda_eeg={lambda_eeg:.3f} lambda_fnirs={lambda_fnirs:.3f} "  # NEW
          f"-> best_val_acc={best_val_acc*100:.2f}% ({elapsed_min:.1f} min)")  # NEW

    return best_val_acc  # NEW: Optuna will try to MAXIMIZE this


if __name__ == "__main__":
    GLOBAL_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # NEW
    print(f"Running on Device: {GLOBAL_DEVICE}\n")  # NEW

    print("Loading train/validation data ONCE (reused across all Optuna trials)...")  # NEW
    _, _, GLOBAL_TRAIN_LOADER, GLOBAL_VAL_LOADER = build_dataloaders()  # NEW

    print(f"\nStarting Optuna search: {N_TRIALS} trials, TPE sampler, "  # NEW
          f"objective = validation accuracy (test set NOT used during search)\n")  # NEW

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))  # NEW: seeded for reproducibility, consistent with RANDOM_SEED used for the train/val/test subject split
    study_t0 = time.time()  # NEW
    study.optimize(objective, n_trials=N_TRIALS)  # NEW
    study_elapsed_min = (time.time() - study_t0) / 60  # NEW

    print("\n" + "=" * 70)  # NEW
    print(f"OPTUNA SEARCH COMPLETE ({study_elapsed_min:.1f} min total, {N_TRIALS} trials)")  # NEW
    print("=" * 70)  # NEW
    print(f"Best value (val accuracy): {study.best_value*100:.2f}%")  # NEW
    print(f"Best params: lambda_eeg={study.best_params['lambda_eeg']:.4f}, "  # NEW
          f"lambda_fnirs={study.best_params['lambda_fnirs']:.4f}")  # NEW

    print("\nAll trials sorted by validation accuracy (best first):")  # NEW
    sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -1, reverse=True)  # NEW
    for t in sorted_trials:  # NEW
        print(f"  Trial {t.number:3d}: val_acc={t.value*100:.2f}% | "  # NEW
              f"lambda_eeg={t.params['lambda_eeg']:.3f}, lambda_fnirs={t.params['lambda_fnirs']:.3f}")  # NEW

    print("\nUse these best lambda_eeg / lambda_fnirs values in the main training script "  # NEW
          "(TSFNet_model_FIXED_v6_paper_loss.py) for the final single training run and test evaluation.")  # NEW

Running on Device: cuda

Loading train/validation data ONCE (reused across all Optuna trials)...
Scanning Train_Set - 18 subjects on Drive D...
VP017 verified: 1026 windows
VP013 verified: 1023 windows
VP010 verified: 1023 windows
VP026 verified: 1023 windows
VP020 verified: 1026 windows
VP007 verified: 1026 windows
VP006 verified: 1026 windows
VP011 verified: 1023 windows
VP016 verified: 1026 windows
VP023 verified: 1026 windows
VP012 verified: 1026 windows
VP019 verified: 1026 windows
VP002 verified: 1026 windows
VP015 verified: 1010 windows
VP024 verified: 1026 windows
VP014 verified: 1016 windows
VP003 verified: 1026 windows
VP018 verified: 1026 windows
Streaming clean data to disk memory-map (Train_Set)...
--> Train_Set Ready on Disk (CLEAN)! Total Windows: 18430 | Labels: [6144 6151 6135]

Scanning Validation_Set - 3 subjects on Drive D...
VP025 verified: 1023 windows
VP005 verified: 1016 windows
VP022 verified: 1026 windows
Streaming clean data to disk memory-map (Validation_Set

[I 2026-07-03 17:30:22,786] A new study created in memory with name: no-name-2fd973ee-d318-49ba-a4bb-73a41bcbc63d


--> Validation_Set Ready on Disk (CLEAN)! Total Windows: 3065 | Labels: [1023 1023 1019]


Starting Optuna search: 20 trials, TPE sampler, objective = validation accuracy (test set NOT used during search)



[I 2026-07-03 17:33:52,448] Trial 0 finished with value: 0.4535073409461664 and parameters: {'lambda_eeg': 0.33090508913552186, 'lambda_fnirs': 0.7630357298074372}. Best is trial 0 with value: 0.4535073409461664.


Trial   0 | lambda_eeg=0.331 lambda_fnirs=0.763 -> best_val_acc=45.35% (3.5 min)


[I 2026-07-03 17:38:07,314] Trial 1 finished with value: 0.4792822185970636 and parameters: {'lambda_eeg': 0.5989954563585539, 'lambda_fnirs': 0.49899386314777744}. Best is trial 1 with value: 0.4792822185970636.


... (15 outputs left out)

[I 2026-07-03 18:20:19,109] Trial 9 finished with value: 0.431973898858075 and parameters: {'lambda_eeg': 0.37395876398158684, 'lambda_fnirs': 0.26842185514853145}. Best is trial 1 with value: 0.4792822185970636.


Trial   9 | lambda_eeg=0.374 lambda_fnirs=0.268 -> best_val_acc=43.20% (8.4 min)


[W 2026-07-03 18:22:18,266] Trial 10 failed with parameters: {'lambda_eeg': 0.7757017341159218, 'lambda_fnirs': 0.40499549109100147} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\SPACE\myenv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\SPACE\AppData\Local\Temp\ipykernel_7604\3402594172.py", line 457, in objective
    best_val_acc, _ = train_one_config(  # NEW
                      ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\SPACE\AppData\Local\Temp\ipykernel_7604\3402594172.py", line 430, in train_one_config
    eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)  # NEW
                           ^^^^^^^^^^^^^^^^
KeyboardInterrupt
[W 2026-07-03 18:22:18,275] Trial 10 failed with value None.


KeyboardInterrupt: 